# Stage 2: Task-Specific AQG Training (Refactored)

**Version**: 2.0 (Clean, no code duplication)

Notebook ini melatih model IndoT5 untuk task AQG spesifik menggunakan modules yang sudah ada.

**Key Improvements**:
- ✅ No code duplication (menggunakan `task_trainer.py`)
- ✅ Clean separation: notebook = orchestration, modules = logic
- ✅ Bug fix applied: `label_pad_token_id=-100`
- ✅ Simplified from ~200 lines → ~50 lines




## 1. Setup Environment

In [1]:
# Install dependencies
!pip install -q transformers peft datasets accelerate
!pip install -q evaluate rouge_score bert_score
!pip install -q --upgrade torchao  # ✅ Tambahkan ini
print('✓ Dependencies installed')


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 103.2 MB/s eta 0:00:00
✓ Dependencies installed


In [2]:
# Cek versi semua library yang terinstall
import importlib
import subprocess
import sys, torch, platform

libs = [
    "transformers",
    "peft", 
    "datasets",
    "accelerate",
    "evaluate",
    "torch",
    "tokenizers",
    "rouge_score",
    "bert_score",
]


print(f"Python:  {sys.version}")
print(f"OS:      {platform.system()}")
print(f"Torch:   {torch.__version__}")
print(f"CUDA:    {torch.cuda.is_available()}")
print(f"\n ")

print("=== Library Versions ===")
for lib in libs:
    try:
        mod = importlib.import_module(lib.replace("-", "_"))
        version = getattr(mod, "__version__", "unknown")
        print(f"  {lib:<20} {version}")
    except ImportError:
        print(f"  {lib:<20} NOT INSTALLED")

# Cek Python version
import sys
print(f"\n  {'python':<20} {sys.version.split()[0]}")

# Cek CUDA
import torch
print(f"  {'cuda available':<20} {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  {'cuda version':<20} {torch.version.cuda}")
    print(f"  {'gpu name':<20} {torch.cuda.get_device_name(0)}")


Python:  3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
OS:      Linux
Torch:   2.10.0+cu128
CUDA:    True

 
=== Library Versions ===
  transformers         5.0.0


  peft                 0.19.1
  datasets             4.0.0
  accelerate           1.13.0
  evaluate             0.4.6
  torch                2.10.0+cu128
  tokenizers           0.22.2
  rouge_score          unknown
  bert_score           0.3.12

  python               3.12.13
  cuda available       True
  cuda version         12.8
  gpu name             Tesla T4


In [3]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
print('✓ Google Drive mounted')

Mounted at /content/drive
✓ Google Drive mounted


In [4]:
# Setup paths and extract source code
import os, sys, zipfile, shutil

DRIVE_ROOT = '/content/drive/MyDrive/dataset_aqg'
sys.path.insert(0, '/content')

# Extract src if not exists
if not os.path.exists('/content/src'):
    shutil.copy(f'{DRIVE_ROOT}/src_finetuned.zip', '/content/src_finetuned.zip')
    with zipfile.ZipFile('/content/src_finetuned.zip', 'r') as z:
        z.extractall('/content/')
    print('✓ src extracted')
else:
    print('✓ src already exists')

print(f'✓ DRIVE_ROOT: {DRIVE_ROOT}')
print(f'✓ sys.path[0]: {sys.path[0]}')

✓ src extracted
✓ DRIVE_ROOT: /content/drive/MyDrive/dataset_aqg
✓ sys.path[0]: /content


In [6]:
# Verify GPU availability
import torch

if not torch.cuda.is_available():
    raise RuntimeError('GPU not available! Go to Runtime > Change runtime type > T4 GPU')

print(f'✓ GPU: {torch.cuda.get_device_name(0)}')
print(f'✓ Memory: {torch.cuda.get_device_properties(0).total_ memory / 1e9:.1f} GB')

SyntaxError: invalid syntax. Perhaps you forgot a comma? (3332075869.py, line 8)

## 2. Load Model with LoRA

**Using**: `src.finetuned.utils.model_loader` (no code duplication!)

In [7]:
from src.finetuned.utils.model_loader import load_model_with_lora, print_model_info

# Load model with LoRA - UPDATED: Using IndoT5 (580M params) instead of IndoNanoT5 (248M)
# IndoNanoT5 was insufficient for complex AQG task
peft_model, tokenizer = load_model_with_lora(
    model_name='LazarusNLP/IndoNanoT5-base',  
    lora_r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=['q', 'v']
)

# Print detailed info
print_model_info(peft_model, tokenizer)

Loading base model: LazarusNLP/IndoNanoT5-base


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/775 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

...

generation_config.json:   0%|          | 0.00/142 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

✓ Base model loaded
✓ LoRA applied: r=8, alpha=16, target=['q', 'v']
  Trainable: 884,736 (0.36%)
  Total:     248,462,592
✓ Model device: cuda:0
  GPU allocated: 1.00 GB

=== Model Information ===
Model type: PeftModelForSeq2SeqLM
Tokenizer: T5Tokenizer
Vocab size: 32000
Pad token: <pad> (ID: 0)
EOS token: </s> (ID: 1)

Parameters:
  Total: 248,462,592
  Trainable: 884,736 (0.36%)
  Frozen: 247,577,856


## 3. Load Dataset

**Using**: `src.finetuned.data.dataset_loader` (reusable module)

In [8]:
from src.finetuned.data.dataset_loader import DatasetLoader

loader = DatasetLoader()
TASK_DIR = '/content/dataset_aqg/dataset-task-spesifc/'

# Copy dataset from Drive if needed
if not os.path.exists(TASK_DIR + 'train.jsonl'):
    drive_task = f'{DRIVE_ROOT}/dataset-task-spesifc'
    os.makedirs(TASK_DIR, exist_ok=True)
    for f in ['train.jsonl', 'validation.jsonl', 'test.jsonl']:
        shutil.copy(f'{drive_task}/{f}', f'{TASK_DIR}{f}')
    print('✓ Dataset copied from Drive')

# Load datasets
train_dataset = loader.load_dataset(TASK_DIR, split='train')
val_dataset = loader.load_dataset(TASK_DIR, split='validation')
test_dataset = loader.load_dataset(TASK_DIR, split='test')

print(f'\nDataset loaded:')
print(f'  Train: {len(train_dataset)} samples')
print(f'  Val:   {len(val_dataset)} samples')
print(f'  Test:  {len(test_dataset)} samples')

✓ Dataset copied from Drive


Generating train split: 0 examples [00:00, ? examples/s]

✓ Loaded 4529 entries from /content/dataset_aqg/dataset-task-spesifc/train.jsonl


Generating train split: 0 examples [00:00, ? examples/s]

✓ Loaded 566 entries from /content/dataset_aqg/dataset-task-spesifc/validation.jsonl


Generating train split: 0 examples [00:00, ? examples/s]

✓ Loaded 567 entries from /content/dataset_aqg/dataset-task-spesifc/test.jsonl

Dataset loaded:
  Train: 4529 samples
  Val:   566 samples
  Test:  567 samples


In [9]:
# Validate and preview dataset
validation_results = loader.validate_dataset(train_dataset)

sample = train_dataset[0]
print('\n=== Sample Entry ===')
print(f"Input: {sample['input']}...")
# print(f"Target: {sample['target']}...")
print(f"Output: {sample['output']}...")  # ✅ Works for v3


✓ Using output field: 'output'

=== Dataset Validation Summary ===
Total Entries: 4529
Duplicate Count: 0
Avg Input Length: 195.65 chars
Avg Target Length: 239.35 chars
Has Metadata: True
✓ No duplicates found

=== Sample Entry ===
Input: buat_soal_pilihan_ganda: Perhatikan kode berikut:
```python
var_mat = [[10, 20],
           [30, 40],
           [50, 60]]
print(var_mat[0][1] + var_mat[2][1])
```
Kode ini menjumlahkan elemen kolom kedua dari baris pertama dan baris terakhir....
Output: question: Perhatikan kode berikut:
```python
var_mat = [[10, 20],
           [30, 40],
           [50, 60]]
print(var_mat[0][1] + var_mat[2][1])
```
Apa output dari kode tersebut?
answer: 80
distractors: 70 | 90 | 60...


In [10]:
# Drop metadata column (required for HuggingFace datasets)
# train_dataset = train_dataset.remove_columns(['metadata'])
# val_dataset = val_dataset.remove_columns(['metadata'])
# test_dataset = test_dataset.remove_columns(['metadata'])

print(f'✓ Metadata dropped')
print(f'  Columns: {train_dataset.column_names}')
print(f'  Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}')

✓ Metadata dropped
  Columns: ['input', 'output', 'metadata']
  Train: 4529 | Val: 566 | Test: 567


## 4. Baseline Evaluation (Pre-Training)

**Using**: `src.finetuned.evaluation` modules

In [11]:
from src.finetuned.evaluation.metrics_calculator import MetricsCalculator
from src.finetuned.evaluation.model_evaluator import ModelEvaluator

metrics_calc = MetricsCalculator()
evaluator = ModelEvaluator(
    model=peft_model,
    tokenizer=tokenizer,
    metrics_calculator=metrics_calc
)

print('Computing baseline metrics (10 samples)...')
baseline_metrics = evaluator.evaluate_on_test_set(
    test_dataset=val_dataset,
    num_beams=4,
    include_bertscore=False,
    max_samples=10
)

print(f"\nBaseline Metrics:")
print(f"  BLEU-4:  {baseline_metrics.get('bleu_4', 0):.4f}")
print(f"  ROUGE-L: {baseline_metrics.get('rouge_l', 0):.4f}")

Computing baseline metrics (10 samples)...

EVALUATING ON TEST SET

Evaluating 10 samples...
  Processed 10/10 samples...
✓ Generated 10 predictions
Computing metrics for 10 samples...
  Computing BLEU...


  Computing ROUGE...


  Computing Diversity...
✓ All metrics computed

Test Set Evaluation Results

BLEU Scores:
  BLEU:     0.0208
  BLEU-1:   0.1012
  BLEU-2:   0.0243
  BLEU-3:   0.0109
  BLEU-4:   0.0069

ROUGE Scores:
  ROUGE-1:  0.1320
  ROUGE-2:  0.0453
  ROUGE-L:  0.1067

Diversity:
  Distinct-1: 0.3149
  Distinct-2: 0.6124


Baseline Metrics:
  BLEU-4:  0.0069
  ROUGE-L: 0.1067


## 5. Configure Training

**Using**: `src.finetuned.training.task_trainer` (with bug fix applied!)

In [12]:
from src.finetuned.training.task_trainer import TaskSpecificTrainer

CHECKPOINT_DIR = '/content/drive/MyDrive/dataset_aqg/checkpoints/04-indonanoot5-report'

# Initialize trainer (all logic in task_trainer.py)
trainer = TaskSpecificTrainer(
    model=peft_model,
    tokenizer=tokenizer,
    output_dir=CHECKPOINT_DIR,
    max_length=512,
    metrics_calculator=metrics_calc
)

print('✓ Trainer initialized')
print(f'  Checkpoints will be saved to: {CHECKPOINT_DIR}')

✓ Trainer initialized
  Checkpoints will be saved to: /content/drive/MyDrive/dataset_aqg/checkpoints/04-indonanoot5-report


## 6. Start Training

**Note**: Training menggunakan default config dari `task_trainer.py`:
- Learning rate: 1e-4
- Batch size: 32 (8×4)
- Warmup: 50 steps
- Epochs: 3
- **Bug fix applied**: `label_pad_token_id=-100` ✅

In [13]:
import time
import os
from pathlib import Path

start_time = time.time()

# Ensure checkpoint directory exists
Path(CHECKPOINT_DIR).mkdir(parents=True, exist_ok=True)

# Check for existing checkpoints
checkpoints = []
if os.path.exists(CHECKPOINT_DIR):
    checkpoints = [d for d in os.listdir(CHECKPOINT_DIR) if d.startswith('checkpoint-')]

# Decide whether to resume
if checkpoints:
    print(f"📂 Found {len(checkpoints)} checkpoint(s): {sorted(checkpoints)}")
    print(f"🔄 Resuming from last checkpoint")
    resume = True
else:
    print("🆕 No checkpoints found - starting fresh training")
    resume = False

print('Starting task-specific AQG training...')
print('='*60)

# Train (all logic in task_trainer.py)
results = trainer.train(
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    early_stopping=False , # ✅ DISABLED: Run full 10 epochs,
    # early_stopping_patience=5,  # Not needed when early_stopping=False
    resume_from_checkpoint=resume  # ✅ Auto-resume jika ada checkpoint
)

elapsed = (time.time() - start_time) / 3600
print(f'\n✓ Training completed in {elapsed:.2f} hours')
print(f'  Final training loss: {results["training_loss"]:.4f}')

Parameter 'function'=<function TaskSpecificTrainer.preprocess_dataset.<locals>.tokenize_function at 0x784392b86e80> of the transform datasets.arrow_dataset.Dataset._map_single couldn't be hashed properly, a random hash was used instead. Make sure your transforms and parameters are serializable with pickle or dill for the dataset fingerprinting and caching to work. If you reuse this transform, the caching mechanism will consider it to be different from the previous calls and recompute everything. This warning is only showed once. Subsequent hashing failures won't be showed.


🆕 No checkpoints found - starting fresh training
Starting task-specific AQG training...

STARTING TASK-SPECIFIC AQG TRAINING

Preprocessing datasets...
Preprocessing 4529 samples...


Tokenizing:   0%|          | 0/4529 [00:00<?, ? examples/s]

✓ Preprocessed 4529 samples
  Note: Padding and label masking will be handled by DataCollatorForSeq2Seq
Preprocessing 566 samples...


Tokenizing:   0%|          | 0/566 [00:00<?, ? examples/s]

...

Epoch,Training Loss,Validation Loss,Bleu 1,Bleu 4,Rouge L
1,19.083123,8.889839,0.000000,0.000000,0.000000
2,17.624692,8.547582,0.000000,0.000000,0.000000
3,17.244860,8.381762,0.002582,0.002582,0.000000
4,16.968055,8.264086,0.003477,0.003477,0.000000
5,16.825050,8.192262,0.006059,0.006059,0.000000
6,16.699106,8.150372,0.002034,0.002034,0.000000
7,16.657468,8.122617,0.000391,0.000391,0.000000
8,16.599529,8.105799,0.000000,0.000000,0.000000
9,16.567992,8.094973,0.000433,0.000433,0.000000
10,16.579165,8.091456,0.000292,0.000292,0.000000


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.12/dist-packages/to


=== Training Complete ===
Final training loss: 17.1182
Training time: 11916.22 seconds
Training samples per second: 3.80


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()



=== Final Evaluation Metrics ===
eval_loss: 8.1923
eval_bleu_1: 0.0061
eval_bleu_4: 0.0061
eval_rouge_l: 0.0000
eval_runtime: 1052.1812
eval_samples_per_second: 0.5380
eval_steps_per_second: 0.0670
✓ Training results saved to /content/drive/MyDrive/dataset_aqg/checkpoints/04-indonanoot5-report/training_results.json

✓ Training completed in 3.60 hours
  Final training loss: 17.1182


## 7. Save Model & Visualize

In [14]:
# Save final model
model_path = trainer.save_final_model(output_name='indot5-python-aqg')
print(f'✓ Model saved to: {model_path}')

# Plot training curves
trainer.plot_training_curves(
    save_path=f'{CHECKPOINT_DIR}/training_curves.png'
)


✓ Final model saved to: /content/drive/MyDrive/dataset_aqg/checkpoints/04-indonanoot5-report/indot5-python-aqg
✓ Model saved to: /content/drive/MyDrive/dataset_aqg/checkpoints/04-indonanoot5-report/indot5-python-aqg
✓ Training curves saved to /content/drive/MyDrive/dataset_aqg/checkpoints/04-indonanoot5-report/training_curves.png


## 8. Final Evaluation

In [15]:
# Re-initialize evaluator with trained model
evaluator_final = ModelEvaluator(
    model=peft_model,
    tokenizer=tokenizer,
    metrics_calculator=metrics_calc
)

print('Running comprehensive evaluation on test set...')
final_metrics = evaluator_final.evaluate_on_test_set(
    test_dataset=test_dataset,
    num_beams=4,
    include_bertscore=True,
    max_samples=None
)

print('\n=== Evaluation Results ===')
for key, value in final_metrics.items():
    print(f'{key}: {value:.4f}')

Running comprehensive evaluation on test set...

EVALUATING ON TEST SET

Evaluating 567 samples...
  Processed 10/567 samples...
  Processed 20/567 samples...
  Processed 30/567 samples...
  Processed 40/567 samples...
  Processed 50/567 samples...
  Processed 60/567 samples...
  Processed 70/567 samples...
  Processed 80/567 samples...
  Processed 90/567 samples...
  Processed 100/567 samples...
  Processed 110/567 samples...
  Processed 120/567 samples...
  Processed 130/567 samples...
  Processed 140/567 samples...
  Processed 150/567 samples...
  Processed 160/567 samples...
  Processed 170/567 samples...
  Processed 180/567 samples...
  Processed 190/567 samples...
  Processed 200/567 samples...
  Processed 210/567 samples...
  Processed 220/567 samples...
  Processed 230/567 samples...
  Processed 240/567 samples...
  Processed 250/567 samples...
  Processed 260/567 samples...
  Processed 270/567 samples...


: 

## 9. Generate Sample Outputs

In [ ]:
EVAL_DIR = '/content/drive/MyDrive/dataset_aqg/evaluation_results/04-indonanoot5-report'

samples = evaluator_final.generate_samples(
    test_dataset=test_dataset,
    num_samples=20,
    num_beams=4,
    save_path=f'{EVAL_DIR}/sample_outputs.json'
)

print(f'✓ {len(samples)} sample outputs generated and saved')
print(f'✓ Full output displayed above with BLEU scores')

: 

## 10. Final Summary

In [ ]:
import json
from pathlib import Path

# Compare with baseline
comparison = evaluator_final.compare_with_baseline(
    finetuned_metrics=final_metrics,
    baseline_metrics=baseline_metrics
)

# Save evaluation report
Path(EVAL_DIR).mkdir(parents=True, exist_ok=True)
report = {
    'baseline_metrics': baseline_metrics,
    'final_metrics': final_metrics,
    'comparison': comparison,
    'training_time_hours': elapsed,
    'model_path': model_path,
    'config': {
        'learning_rate': 1e-4,
        'batch_size': 32,
        'epochs': 3,
        'lora_r': 8,
        'lora_alpha': 16
    }
}

with open(f'{EVAL_DIR}/evaluation_report.json', 'w') as f:
    json.dump(report, f, indent=2, default=str)

# Print summary
print('\n' + '='*60)
print('TASK-SPECIFIC AQG TRAINING SUMMARY')
print('='*60)
print(f'Training Time: {elapsed:.2f} hours')
print(f'Model saved: {model_path}')
print(f'\nMetrics Comparison:')
print(f"  BLEU-4:       {baseline_metrics.get('bleu_4',0):.4f} → {final_metrics.get('bleu_4',0):.4f}")
print(f"  ROUGE-L:      {baseline_metrics.get('rouge_l',0):.4f} → {final_metrics.get('rouge_l',0):.4f}")
print(f"  BERTScore F1: {baseline_metrics.get('bertscore_f1',0):.4f} → {final_metrics.get('bertscore_f1',0):.4f}")

bleu_improvement = comparison.get('bleu_4_improvement_pct', 0)
print(f'\nBLEU-4 Improvement: {bleu_improvement:+.1f}%')

if final_metrics.get('bleu_4', 0) >= 0.35:
    print('\n✓ SUCCESS: BLEU-4 target achieved (>= 0.35)')
else:
    print(f"\n⚠ BLEU-4 = {final_metrics.get('bleu_4',0):.4f} (target: >= 0.35)")
    print('  Consider: more epochs, lower lr, or larger dataset')

print('\n✓ Fine-tuning pipeline complete!')
print(f'  Evaluation report: {EVAL_DIR}/evaluation_report.json')
print(f'  Sample outputs: {EVAL_DIR}/sample_outputs.json')